In [ ]:
import os, glob

print(os.getcwd())
print(os.path.exists("../data/raw"))
print(glob.glob("../data/raw/*.csv")[:3])

In [ ]:
from pathlib import Path
import os
import pandas as pd

root = Path.cwd()
while not (root / "data" / "raw").exists() and root != root.parent:
    root = root.parent
raw_dir = root / "data" / "raw"
print(raw_dir)

frames = []
for path in sorted(raw_dir.glob("[0-9][0-9][0-9][0-9].csv")):
    d = pd.read_csv(path)
    d["file_year"] = int(path.stem)
    frames.append(d)

print(len(frames), "files loaded")
raw_all = pd.concat(frames, ignore_index=True)
print(raw_all.shape)
print(raw_all.groupby("file_year").size())

In [ ]:
reference = list(pd.read_csv(raw_dir / "2024.csv", nrows=0).columns)

for f in frames:
    year = f["file_year"].iloc[0]
    cols_f = [c for c in f.columns if c != "file_year"]
    if cols_f != reference:
        print(year, "| missing:", set(reference) - set(cols_f), "| extra:", set(cols_f) - set(reference), "| same set, different order:", set(cols_f) == set(reference))

print("column check done")

In [ ]:
cols = ["minutes", "w_ace", "w_svpt", "indoor"]
print(raw_all[cols].isna().groupby(raw_all["file_year"]).mean().round(3).to_string())

In [ ]:
print(pd.crosstab(raw_all["file_year"], raw_all["tourney_level"]).to_string())

In [ ]:
cols = ["minutes", "w_ace", "w_svpt", "indoor"]
nd = raw_all[raw_all["tourney_level"] != "D"]
print(nd[cols].isna().groupby(nd["file_year"]).mean().round(3).to_string())

In [ ]:
sub = nd[nd["file_year"].between(2013, 2017)]
print(sub["w_ace"].isna().groupby([sub["file_year"], sub["tourney_level"]]).mean().round(3).unstack().to_string())

In [ ]:
a = raw_all[raw_all["tourney_level"] == "A"]
print(a.groupby("tourney_name")["file_year"].agg(["min", "max", "count"]).to_string())

m = raw_all[(raw_all["tourney_level"] == "M") & (raw_all["file_year"] >= 2015)]
print(pd.crosstab(m["tourney_name"], m["file_year"]).to_string())

In [ ]:
x = raw_all[raw_all["tourney_level"].isin(["G", "M", "F"])]
print(x.groupby(["tourney_level", "tourney_name"])["file_year"].agg(["min", "max", "count"]).to_string())

In [ ]:
#Changed surface?
x = raw_all[raw_all["tourney_lesvel"].isin(["G", "M", "F"])]
print(x.groupby(["tourney_name", "surface", "indoor"], dropna=False)["file_year"].agg(["min", "max", "count"]).to_string())

In [ ]:
t = raw_all[raw_all["tourney_level"].isin(["G", "M", "F"]) & (raw_all["file_year"] >= 2023)]
print(t.groupby(["tourney_name", "file_year"])["tourney_id"].first().unstack().to_string())

In [ ]:
odd = raw_all[
    ((raw_all["tourney_name"] == "Rome Masters") & raw_all["file_year"].isin([2020, 2021, 2022]))
    | ((raw_all["tourney_name"] == "Canada Masters") & raw_all["file_year"].isin([2021, 2022]))
]
print(pd.crosstab([odd["tourney_name"], odd["file_year"]], odd["round"]).to_string())

In [ ]:
#duplicates?
print(raw_all.duplicated(subset=["tourney_id", "match_num", "file_year"]).sum())

In [ ]:
print("same tournament + match number:", raw_all.duplicated(subset=["tourney_id", "match_num", "file_year"]).sum())
print("completely identical rows:", raw_all.duplicated().sum())
print("empty match_num:", raw_all["match_num"].isna().sum())

In [ ]:
d = raw_all[raw_all.duplicated(subset=["tourney_id", "match_num", "file_year"], keep=False)]
print(len(d))
print(pd.crosstab(d["file_year"], d["tourney_level"]).to_string())
print(d["tourney_name"].value_counts().head(10).to_string())

In [ ]:
for col in ["indoor", "surface", "best_of"]:
    print(raw_all[col].value_counts(dropna=False).to_string())
    print()

In [ ]:
x = raw_all[raw_all["tourney_level"].isin(["G", "M", "F"])].copy()
x["event_no"] = x["tourney_id"].astype(str).str.split("-").str[1]
print(x.groupby(["event_no", "tourney_name"])["file_year"].agg(["min", "max", "count"]).to_string())

In [ ]:
key = ["tourney_id", "round", "winner_id", "loser_id"]
mm = raw_all[raw_all["match_num"].isna()]
print(mm[key].isna().sum().to_string())
print("repeats of that key, whole dataset:", raw_all.duplicated(subset=key + ["file_year"]).sum())

In [ ]:
s = raw_all[raw_all["surface"].isna()]
print(s.groupby(["file_year", "tourney_level"]).size().to_string())
print(s["tourney_name"].value_counts().head(10).to_string())

In [ ]:
nums = ["580","520","540","560","404","403","410","1536","416","421","422","5014","352","605"]
y = raw_all.copy()
y["event_no"] = y["tourney_id"].astype(str).str.split("-").str[1]
z = y[y["event_no"].isin(nums)]
print(z.groupby(["event_no", "tourney_level", "tourney_name"])["file_year"].agg(["min", "max", "count"]).to_string())

In [ ]:
key = ["tourney_id", "round", "winner_id", "loser_id"]
r = raw_all[raw_all.duplicated(subset=key + ["file_year"], keep=False)]
print(len(r))
print(r.groupby(["file_year", "tourney_level"]).size().to_string())
print(r["tourney_name"].value_counts().head(10).to_string())
print("identical apart from match_num:", raw_all.drop(columns=["match_num"]).duplicated().sum())

In [ ]:
t = pd.read_csv(root / "data" / "own" / "tournaments.csv")
print(t.shape)
print(t[["event_no", "tournament_id", "level"]].to_string())

In [ ]:
key = ["tourney_id", "round", "winner_id", "loser_id"]
r = raw_all[raw_all.duplicated(subset=key + ["file_year"], keep=False) & (raw_all["tourney_level"] != "D")]
print(r.sort_values(key)[["file_year", "tourney_name", "round", "match_num", "score"]].to_string())